# 🏀 Interactive NBA Player Explorer

**Explore any NBA player's shooting, play style, trajectory, and head-to-head comparisons — all in one interactive dashboard.**

*Part 6 of 10 in the [nbadb](https://github.com/wyattowalsh/nbadb) notebook suite* | [Dataset on Kaggle](https://www.kaggle.com/datasets/wyattowalsh/basketball)

---

## Table of Contents

1. [Setup & Helpers](#1-setup--helpers)
2. [Player Bio Cards](#2-player-bio-cards)
3. [Shot Charts](#3-shot-charts)
4. [Synergy Play Type Profiles](#4-synergy-play-type-profiles)
5. [Percentile Radar](#5-percentile-radar)
6. [Season Trajectory](#6-season-trajectory)
7. [Head-to-Head Comparisons](#7-head-to-head-comparisons)
8. [Cleanup & Cross-Links](#8-cleanup--cross-links)

In [ ]:
!pip install -q "duckdb>=1.4,<2" "polars>=1.38,<2" "plotly>=5.18,<6" "scikit-learn>=1.4,<2"

In [ ]:
import sys
from pathlib import Path

import duckdb
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# Import shared utilities
_kaggle_path = Path("/kaggle/input/basketball")
sys.path.insert(0, str(_kaggle_path if _kaggle_path.exists() else Path(".")))
from nbadb_utils import COLORS, TEMPLATE, takeaway, draw_court_plotly, get_connection

conn = get_connection()

# --- Detect current season ---
CURRENT_SEASON = conn.sql(
    "SELECT MAX(season_year) FROM dim_game"
).fetchone()[0]
print(f"Current season: {CURRENT_SEASON}")

## 1. Setup & Helpers

Below we define three reusable helpers that other notebooks in this suite also rely on:

- **`draw_court_plotly()`** — returns Plotly layout shapes for an NBA half-court
- **`make_radar()`** — builds a single `Scatterpolar` trace for radar/spider charts
- **`percentile_rank()`** — computes position-relative percentile rankings across 10 dimensions

In [ ]:
# draw_court_plotly() is imported from nbadb_utils — quick visual test
fig = go.Figure()
fig.update_layout(
    shapes=draw_court_plotly(),
    xaxis=dict(range=[-260, 260], showgrid=False, zeroline=False, visible=False),
    yaxis=dict(range=[-60, 440], showgrid=False, zeroline=False, visible=False,
               scaleanchor="x", scaleratio=1),
    template=TEMPLATE, height=500, width=470, margin=dict(l=20, r=20, t=20, b=20),
    plot_bgcolor="white",
)
fig.show()

In [ ]:
def make_radar(categories, values, name, color):
    """Return a go.Scatterpolar trace for a radar/spider chart.

    Parameters
    ----------
    categories : list[str]
        Axis labels (e.g., ["Scoring", "Playmaking", ...]).
    values : list[float]
        Values for each category (same length as categories).
    name : str
        Trace legend name.
    color : str
        CSS color string.
    """
    return go.Scatterpolar(
        r=values + [values[0]],  # close the polygon
        theta=categories + [categories[0]],
        fill="toself",
        fillcolor=color.replace(")", ", 0.15)").replace("rgb", "rgba")
            if color.startswith("rgb") else color + "26",
        line=dict(color=color, width=2),
        name=name,
        hovertemplate="%{theta}: %{r:.0f}<extra></extra>",
    )

In [ ]:
# Dimension definitions for percentile_rank
RADAR_DIMENSIONS = [
    "Scoring", "Playmaking", "Rebounding", "Efficiency",
    "Hustle", "Defense", "Speed", "Volume", "3PT Shooting", "Passing",
]


def percentile_rank(df, player_id, position, dimensions=None):
    """Compute percentile rank (0-100) for a player vs same-position peers.

    Parameters
    ----------
    df : polars.DataFrame
        Must contain columns from agg_player_season + analytics aggregates.
    player_id : int
        Target player.
    position : str
        Position filter (e.g., "Guard", "Forward", "Center").
    dimensions : list[str] | None
        Which RADAR_DIMENSIONS to compute. Defaults to all 10.

    Returns
    -------
    dict[str, float]
        {dimension_name: percentile_value}
    """
    if dimensions is None:
        dimensions = RADAR_DIMENSIONS

    # Map dimension names to column names
    dim_to_col = {
        "Scoring": "avg_pts",
        "Playmaking": "avg_ast",
        "Rebounding": "avg_reb",
        "Efficiency": "avg_ts_pct",
        "Hustle": "contested_shots_pg",
        "Defense": "def_combined",
        "Speed": "spd_pg",
        "Volume": "total_min",
        "3PT Shooting": "fg3_pct",
        "Passing": "passes_pg",
    }

    # Simplify position to broad category for comparison
    pos_str = (position or "").upper()
    if "G" in pos_str and "F" not in pos_str and "C" not in pos_str:
        pos_group = "Guard"
    elif "C" in pos_str and "F" not in pos_str and "G" not in pos_str:
        pos_group = "Center"
    else:
        pos_group = "Forward"

    # Filter peers by broad position group
    peers = df.filter(
        pl.col("pos_group") == pos_group
    )

    player_row = peers.filter(pl.col("player_id") == player_id)
    if player_row.is_empty():
        import warnings
        warnings.warn(f"percentile_rank: player_id={player_id} not found in peer group; returning median (50.0) for all dimensions", stacklevel=2)
        return {d: 50.0 for d in dimensions}

    result = {}
    for dim in dimensions:
        col = dim_to_col.get(dim)
        if col is None or col not in peers.columns:
            result[dim] = 50.0
            continue
        val = player_row[col][0]
        if val is None:
            result[dim] = 50.0
            continue
        col_vals = peers[col].drop_nulls()
        if len(col_vals) == 0:
            result[dim] = 50.0
            continue
        pctile = (col_vals < val).sum() / len(col_vals) * 100
        result[dim] = round(float(pctile), 1)

    return result

---

## 2. Player Bio Cards

Who are the most-played players this season? Let's pull the top 10 by total minutes and build styled profile cards showing their key biographical info and career context.

In [ ]:
top_players = conn.sql(f"""
    SELECT
        s.player_id,
        p.full_name,
        p.position,
        s.team_id,
        p.height,
        p.weight,
        p.birth_date,
        p.country,
        p.draft_year,
        p.draft_round,
        p.draft_number,
        p.from_year,
        p.to_year,
        s.gp,
        s.total_min,
        s.avg_pts,
        s.avg_reb,
        s.avg_ast,
        s.avg_ts_pct,
        s.fg3_pct
    FROM agg_player_season s
    JOIN dim_player p ON s.player_id = p.player_id AND p.is_current = TRUE
    WHERE s.season_year = '{CURRENT_SEASON}'
      AND s.season_type = 'Regular Season'
    ORDER BY s.total_min DESC
    LIMIT 10
""").pl()

print(f"Top 10 players by minutes in {CURRENT_SEASON}:")
top_players.select(["full_name", "position", "gp", "total_min", "avg_pts", "avg_reb", "avg_ast"])

In [ ]:
# Render styled bio cards
cards_html = '<div style="display:flex; flex-wrap:wrap; gap:16px; margin:16px 0;">'

for row in top_players.iter_rows(named=True):
    draft_info = (
        f"R{row['draft_round']} Pick {row['draft_number']} ({row['draft_year']})"
        if row["draft_year"] and row["draft_year"] > 0
        else "Undrafted"
    )
    career_span = f"{row['from_year']}-{row['to_year']}" if row["from_year"] else "N/A"
    ts_display = f"{row['avg_ts_pct']:.1%}" if row["avg_ts_pct"] else "N/A"
    fg3_display = f"{row['fg3_pct']:.1%}" if row["fg3_pct"] else "N/A"

    cards_html += f"""
    <div style="background:white; border:1px solid #ddd; border-radius:8px;
                padding:16px; width:280px; box-shadow:0 2px 4px rgba(0,0,0,0.1);">
        <div style="font-size:16px; font-weight:700; color:{COLORS['primary']};">
            {row['full_name']}
        </div>
        <div style="color:{COLORS['neutral']}; font-size:13px; margin-bottom:8px;">
            {row['position']} &middot; Team {row['team_id']}
        </div>
        <table style="font-size:13px; width:100%; border-collapse:collapse;">
            <tr><td style="color:{COLORS['neutral']};">Height / Weight</td>
                <td style="text-align:right;">{row['height']} / {row['weight']} lbs</td></tr>
            <tr><td style="color:{COLORS['neutral']};">Draft</td>
                <td style="text-align:right;">{draft_info}</td></tr>
            <tr><td style="color:{COLORS['neutral']};">Career</td>
                <td style="text-align:right;">{career_span}</td></tr>
            <tr><td style="color:{COLORS['neutral']};">Country</td>
                <td style="text-align:right;">{row['country']}</td></tr>
        </table>
        <hr style="border:none; border-top:1px solid #eee; margin:8px 0;">
        <div style="display:flex; justify-content:space-around; text-align:center; font-size:13px;">
            <div><div style="font-weight:700; color:{COLORS['secondary']};">{row['avg_pts']:.1f}</div>PPG</div>
            <div><div style="font-weight:700; color:{COLORS['primary']};">{row['avg_reb']:.1f}</div>RPG</div>
            <div><div style="font-weight:700; color:{COLORS['primary']};">{row['avg_ast']:.1f}</div>APG</div>
            <div><div style="font-weight:700; color:{COLORS['accent']};">{ts_display}</div>TS%</div>
        </div>
    </div>
    """

cards_html += "</div>"
display(HTML(cards_html))

In [ ]:
takeaway(
    "The heaviest-minute players tend to be primary ball-handlers and franchise cornerstones. "
    "Note the range of True Shooting percentages — volume and efficiency rarely go hand in hand."
)

---

## 3. Shot Charts

Shot charts reveal **where** a player operates on the floor. Green dots are makes, red are misses. Use the dropdown to switch between the top 20 scorers and see how each player's shot selection differs.

Players who cluster near the rim tend to be finishers; players with broad arc coverage are perimeter threats.

In [ ]:
# Get top-20 scorers for the current season
top_scorers = conn.sql(f"""
    SELECT s.player_id, p.full_name, s.avg_pts
    FROM agg_player_season s
    JOIN dim_player p ON s.player_id = p.player_id AND p.is_current = TRUE
    WHERE s.season_year = '{CURRENT_SEASON}'
      AND s.season_type = 'Regular Season'
      AND s.gp >= 20
    ORDER BY s.avg_pts DESC
    LIMIT 20
""").pl()

scorer_ids = top_scorers["player_id"].to_list()
scorer_names = top_scorers["full_name"].to_list()
scorer_id_str = ", ".join(str(x) for x in scorer_ids)

# Get shot chart data
shots = conn.sql(f"""
    SELECT
        sc.player_id, p.full_name AS player_name,
        sc.loc_x, sc.loc_y,
        sc.shot_made_flag, sc.action_type,
        sc.shot_distance, sc.period
    FROM fact_shot_chart sc
    JOIN dim_game g ON sc.game_id = g.game_id
    JOIN dim_player p ON sc.player_id = p.player_id AND p.is_current = TRUE
    WHERE g.season_year = '{CURRENT_SEASON}'
      AND sc.player_id IN ({scorer_id_str})
""").pl()

print(f"Loaded {len(shots):,} shots for {len(scorer_ids)} players")

In [ ]:
fig = go.Figure()

# Create two traces per player (made + missed) for coloring
for i, (pid, name) in enumerate(zip(scorer_ids, scorer_names)):
    player_shots = shots.filter(pl.col("player_id") == pid)

    for made, color, symbol, label in [
        (1, COLORS["positive"], "circle", "Made"),
        (0, COLORS["negative"], "x", "Missed"),
    ]:
        subset = player_shots.filter(pl.col("shot_made_flag") == made)
        fig.add_trace(go.Scatter(
            x=subset["loc_x"].to_list(),
            y=subset["loc_y"].to_list(),
            mode="markers",
            marker=dict(color=color, size=4, symbol=symbol, opacity=0.6),
            name=f"{name} ({label})",
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Action: %{customdata[1]}<br>"
                "Distance: %{customdata[2]} ft<br>"
                "Period: %{customdata[3]}<extra></extra>"
            ),
            customdata=[
                [name, row["action_type"], row["shot_distance"], row["period"]]
                for row in subset.iter_rows(named=True)
            ],
            visible=(i == 0),  # only first player visible by default
        ))

# Build updatemenus dropdown
buttons = []
for i, name in enumerate(scorer_names):
    # Each player has 2 traces (made + missed)
    visibility = [False] * (len(scorer_names) * 2)
    visibility[i * 2] = True
    visibility[i * 2 + 1] = True
    buttons.append(dict(
        label=name,
        method="update",
        args=[{"visible": visibility}],
    ))

fig.update_layout(
    shapes=draw_court_plotly(),
    updatemenus=[dict(
        type="dropdown",
        direction="down",
        x=0.02, y=0.98,
        xanchor="left", yanchor="top",
        buttons=buttons,
        bgcolor="white",
        bordercolor=COLORS["primary"],
        font=dict(size=12),
    )],
    xaxis=dict(range=[-260, 260], showgrid=False, zeroline=False, visible=False),
    yaxis=dict(range=[-60, 440], showgrid=False, zeroline=False, visible=False,
               scaleanchor="x", scaleratio=1),
    template=TEMPLATE,
    height=600, width=550,
    margin=dict(l=20, r=20, t=40, b=20),
    plot_bgcolor="white",
    title=dict(text="Shot Chart — Select Player", font=dict(size=16, color=COLORS["primary"])),
    showlegend=False,
)
fig.show()

In [ ]:
takeaway(
    "Shot selection is a fingerprint. Bigs cluster around the rim and short mid-range; "
    "guards show arcs of three-pointers and pull-up jumpers. "
    "Switch between players to see how dramatically shot profiles differ."
)

---

## 4. Synergy Play Type Profiles

NBA Synergy data tells us **how** a player scores — transition, isolation, pick-and-roll, spot-up, post-up, etc. Below we show each player's possession distribution by play type, colored by whether they score **above** (green) or **below** (red) the league average points per possession for that play type.

In [ ]:
# Get top-10 player IDs (by minutes) for synergy
top10_ids = top_players["player_id"].to_list()
top10_names_map = dict(zip(
    top_players["player_id"].to_list(),
    top_players["full_name"].to_list(),
))
top10_id_str = ", ".join(str(x) for x in top10_ids)

synergy = conn.sql(f"""
    SELECT
        sy.player_id, sy.play_type, sy.poss, sy.ppp,
        SUM(sy.poss) OVER (PARTITION BY sy.player_id) AS total_poss
    FROM fact_synergy sy
    WHERE sy.season_year = '{CURRENT_SEASON}'
      AND sy.type_grouping = 'offensive'
      AND sy.player_id IN ({top10_id_str})
    ORDER BY sy.player_id, sy.poss DESC
""").pl()

# League average PPP per play type
league_avg_ppp = conn.sql(f"""
    SELECT play_type, SUM(pts) * 1.0 / NULLIF(SUM(poss), 0) AS avg_ppp
    FROM fact_synergy
    WHERE season_year = '{CURRENT_SEASON}'
      AND type_grouping = 'offensive'
    GROUP BY play_type
""").pl()

avg_ppp_map = dict(zip(
    league_avg_ppp["play_type"].to_list(),
    league_avg_ppp["avg_ppp"].to_list(),
))

print(f"Synergy data: {len(synergy):,} rows across {synergy['player_id'].n_unique()} players")
print(f"Play types tracked: {sorted(avg_ppp_map.keys())}")

In [ ]:
# Build 2x5 subplot grid
synergy_players = [pid for pid in top10_ids if pid in synergy["player_id"].to_list()]
n_players = len(synergy_players)
n_cols = 5
n_rows = (n_players + n_cols - 1) // n_cols

subplot_titles = [top10_names_map.get(pid, str(pid)) for pid in synergy_players]
fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.06, vertical_spacing=0.15,
)

for idx, pid in enumerate(synergy_players):
    row = idx // n_cols + 1
    col = idx % n_cols + 1

    player_syn = synergy.filter(pl.col("player_id") == pid).sort("poss", descending=True).head(8)
    if player_syn.is_empty():
        continue

    total = player_syn["total_poss"][0] or 1
    play_types = player_syn["play_type"].to_list()
    poss_pcts = [(p / total * 100) for p in player_syn["poss"].to_list()]
    ppps = player_syn["ppp"].to_list()

    bar_colors = [
        COLORS["positive"] if (ppp or 0) >= avg_ppp_map.get(pt, 0.95)
        else COLORS["negative"]
        for pt, ppp in zip(play_types, ppps)
    ]

    fig.add_trace(
        go.Bar(
            y=play_types[::-1],
            x=poss_pcts[::-1],
            orientation="h",
            marker_color=bar_colors[::-1],
            hovertemplate="%{y}: %{x:.1f}% poss<extra></extra>",
            showlegend=False,
        ),
        row=row, col=col,
    )

fig.update_layout(
    height=300 * n_rows,
    width=1100,
    template=TEMPLATE,
    title=dict(
        text="Play Type Profiles — Green = Above League Avg PPP, Red = Below",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    margin=dict(l=20, r=20, t=80, b=20),
    showlegend=False,
)
fig.update_xaxes(showticklabels=False)
fig.show()

In [ ]:
takeaway(
    "Play type distribution separates archetypes. Isolation-heavy scorers, pick-and-roll maestros, "
    "and spot-up snipers each show distinct profiles. Green bars show where a player is creating "
    "above-average value per possession — that is their bread and butter."
)

---

## 5. Percentile Radar

How does a player rank compared to others at the same position? This radar chart shows percentile ranks across 10 dimensions — from scoring and playmaking to hustle and speed. A player at the 90th percentile in Scoring is a better scorer than 90% of same-position peers.

Use the dropdown to explore different players.

In [ ]:
# Build enriched stats DataFrame for percentile computation
# Combines agg_player_season with per-game hustle/tracking averages
radar_df = conn.sql(f"""
    WITH season_stats AS (
        SELECT
            s.player_id, s.team_id, s.gp, s.total_min,
            s.avg_pts, s.avg_reb, s.avg_ast, s.avg_stl, s.avg_blk,
            s.fg3_pct, s.avg_ts_pct,
            p.position, p.full_name
        FROM agg_player_season s
        JOIN dim_player p ON s.player_id = p.player_id AND p.is_current = TRUE
        WHERE s.season_year = '{CURRENT_SEASON}'
          AND s.season_type = 'Regular Season'
          AND s.gp >= 20
    ),
    hustle_tracking AS (
        SELECT
            player_id,
            AVG(contested_shots) AS contested_shots_pg,
            AVG(spd) AS spd_pg,
            AVG(passes) AS passes_pg
        FROM analytics_player_game_complete
        WHERE season_year = '{CURRENT_SEASON}'
        GROUP BY player_id
    )
    SELECT
        ss.*,
        (ss.avg_stl + ss.avg_blk) AS def_combined,
        ht.contested_shots_pg,
        ht.spd_pg,
        ht.passes_pg,
        CASE
            WHEN UPPER(ss.position) LIKE '%G%'
                 AND UPPER(ss.position) NOT LIKE '%F%'
                 AND UPPER(ss.position) NOT LIKE '%C%' THEN 'Guard'
            WHEN UPPER(ss.position) LIKE '%C%'
                 AND UPPER(ss.position) NOT LIKE '%F%'
                 AND UPPER(ss.position) NOT LIKE '%G%' THEN 'Center'
            ELSE 'Forward'
        END AS pos_group
    FROM season_stats ss
    LEFT JOIN hustle_tracking ht ON ss.player_id = ht.player_id
    ORDER BY ss.total_min DESC
""").pl()

print(f"Radar dataset: {len(radar_df)} players ({radar_df['pos_group'].value_counts()})")

In [ ]:
# Take top 50 by minutes for the dropdown
radar_top50 = radar_df.head(50)

fig = go.Figure()

for i, row in enumerate(radar_top50.iter_rows(named=True)):
    pctiles = percentile_rank(
        radar_df, row["player_id"], row["position"], RADAR_DIMENSIONS
    )
    values = [pctiles[d] for d in RADAR_DIMENSIONS]
    fig.add_trace(make_radar(
        RADAR_DIMENSIONS, values, row["full_name"], COLORS["primary"]
    ))
    fig.data[-1].visible = (i == 0)

# Build dropdown buttons
buttons = []
for i, row in enumerate(radar_top50.iter_rows(named=True)):
    vis = [False] * len(radar_top50)
    vis[i] = True
    buttons.append(dict(
        label=row["full_name"],
        method="update",
        args=[{"visible": vis}],
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(range=[0, 100], showticklabels=True, tickfont=dict(size=10)),
        angularaxis=dict(tickfont=dict(size=11)),
    ),
    updatemenus=[dict(
        type="dropdown",
        direction="down",
        x=0.02, y=0.98,
        xanchor="left", yanchor="top",
        buttons=buttons,
        bgcolor="white",
        bordercolor=COLORS["primary"],
        font=dict(size=12),
    )],
    template=TEMPLATE,
    height=550, width=650,
    margin=dict(l=60, r=60, t=40, b=40),
    title=dict(
        text="Percentile Radar — Select Player",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    showlegend=False,
)
fig.show()

In [ ]:
takeaway(
    "Radar charts expose well-rounded stars vs specialists. Elite two-way players fill most of the "
    "polygon; pure shooters spike on 3PT and Efficiency but drop on Rebounding and Hustle. "
    "Percentiles are position-relative, so guards and centers are each compared to their peers."
)

---

## 6. Season Trajectory

Rolling averages smooth out game-to-game noise and reveal **trends** — is a player peaking, slumping, or steady? Below we plot 10-game rolling averages for points, rebounds, and assists across the season.

In [ ]:
rolling = conn.sql(f"""
    SELECT
        r.player_id, r.game_date,
        r.pts_roll10, r.reb_roll10, r.ast_roll10,
        p.full_name
    FROM agg_player_rolling r
    JOIN dim_game g ON r.game_id = g.game_id
    JOIN dim_player p ON r.player_id = p.player_id AND p.is_current = TRUE
    WHERE g.season_year = '{CURRENT_SEASON}'
      AND r.player_id IN ({top10_id_str})
    ORDER BY r.player_id, r.game_date
""").pl()

rolling_players = (
    rolling.select(["player_id", "full_name"])
    .unique()
    .sort("player_id")
)
rolling_names = rolling_players["full_name"].to_list()
rolling_pids = rolling_players["player_id"].to_list()

print(f"Rolling data: {len(rolling):,} game-rows for {len(rolling_names)} players")

In [ ]:
fig = go.Figure()

stat_colors = {
    "pts_roll10": (COLORS["secondary"], "Points"),
    "reb_roll10": (COLORS["primary"], "Rebounds"),
    "ast_roll10": (COLORS["accent"], "Assists"),
}

# 3 traces per player (pts, reb, ast)
traces_per_player = 3

for i, (pid, name) in enumerate(zip(rolling_pids, rolling_names)):
    p_data = rolling.filter(pl.col("player_id") == pid)
    dates = p_data["game_date"].to_list()

    for stat_col, (color, stat_label) in stat_colors.items():
        fig.add_trace(go.Scatter(
            x=dates,
            y=p_data[stat_col].to_list(),
            mode="lines",
            line=dict(color=color, width=2),
            name=stat_label,
            hovertemplate=f"{stat_label}: " + "%{y:.1f}<extra></extra>",
            visible=(i == 0),
            showlegend=(i == 0),
        ))

# Dropdown
buttons = []
for i, name in enumerate(rolling_names):
    vis = [False] * (len(rolling_names) * traces_per_player)
    for j in range(traces_per_player):
        vis[i * traces_per_player + j] = True
    buttons.append(dict(
        label=name,
        method="update",
        args=[{"visible": vis}],
    ))

fig.update_layout(
    updatemenus=[dict(
        type="dropdown",
        direction="down",
        x=0.02, y=0.98,
        xanchor="left", yanchor="top",
        buttons=buttons,
        bgcolor="white",
        bordercolor=COLORS["primary"],
        font=dict(size=12),
    )],
    template=TEMPLATE,
    height=450, width=900,
    xaxis_title="Game Date",
    yaxis_title="10-Game Rolling Average",
    title=dict(
        text="Season Trajectory — 10-Game Rolling Averages",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    margin=dict(l=50, r=20, t=60, b=50),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

In [ ]:
takeaway(
    "Rolling averages expose hot streaks, injuries, and second-half surges that raw per-game "
    "numbers obscure. Look for inflection points — they often coincide with lineup changes, "
    "trade-deadline acquisitions, or return from injury."
)

---

## 7. Head-to-Head Comparisons

Every great player has a rival. Below we compare six pairs of top players side by side using percentile radar charts. This reveals complementary strengths and weaknesses — where one player dominates, the other may lag, and vice versa.

In [ ]:
# Build matchup pairs from the top players we already have
# We pair them sequentially from the top-minutes list (1 vs 2, 3 vs 4, etc.)
all_top = radar_df.head(12)  # ensure we have enough for 6 pairs
matchup_pairs = []
for i in range(0, min(12, len(all_top)), 2):
    if i + 1 < len(all_top):
        p1 = all_top.row(i, named=True)
        p2 = all_top.row(i + 1, named=True)
        matchup_pairs.append((p1, p2))

print("Matchup pairs:")
for p1, p2 in matchup_pairs:
    print(f"  {p1['full_name']} vs {p2['full_name']}")

In [ ]:
n_pairs = len(matchup_pairs)
fig = make_subplots(
    rows=2, cols=3,
    specs=[[{"type": "polar"}] * 3] * 2,
    subplot_titles=[
        f"{p1['full_name']} vs {p2['full_name']}" for p1, p2 in matchup_pairs
    ],
    horizontal_spacing=0.08, vertical_spacing=0.12,
)

for idx, (p1, p2) in enumerate(matchup_pairs):
    row = idx // 3 + 1
    col = idx % 3 + 1

    pctiles1 = percentile_rank(radar_df, p1["player_id"], p1["position"])
    pctiles2 = percentile_rank(radar_df, p2["player_id"], p2["position"])

    vals1 = [pctiles1[d] for d in RADAR_DIMENSIONS]
    vals2 = [pctiles2[d] for d in RADAR_DIMENSIONS]

    # Player 1 trace
    fig.add_trace(
        go.Scatterpolar(
            r=vals1 + [vals1[0]],
            theta=RADAR_DIMENSIONS + [RADAR_DIMENSIONS[0]],
            fill="toself",
            fillcolor=COLORS["primary"] + "26",
            line=dict(color=COLORS["primary"], width=2),
            name=p1["full_name"],
            hovertemplate="%{theta}: %{r:.0f}<extra></extra>",
            showlegend=(idx == 0),
        ),
        row=row, col=col,
    )

    # Player 2 trace
    fig.add_trace(
        go.Scatterpolar(
            r=vals2 + [vals2[0]],
            theta=RADAR_DIMENSIONS + [RADAR_DIMENSIONS[0]],
            fill="toself",
            fillcolor=COLORS["secondary"] + "26",
            line=dict(color=COLORS["secondary"], width=2),
            name=p2["full_name"],
            hovertemplate="%{theta}: %{r:.0f}<extra></extra>",
            showlegend=(idx == 0),
        ),
        row=row, col=col,
    )

fig.update_polars(radialaxis=dict(range=[0, 100], showticklabels=False))
fig.update_layout(
    height=700, width=1100,
    template=TEMPLATE,
    title=dict(
        text="Head-to-Head Percentile Radar Comparisons",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    margin=dict(l=40, r=40, t=80, b=40),
    legend=dict(orientation="h", yanchor="bottom", y=-0.05, xanchor="center", x=0.5),
)
fig.show()

In [ ]:
# Comparison table
comparison_rows = []
for p1, p2 in matchup_pairs:
    pctiles1 = percentile_rank(radar_df, p1["player_id"], p1["position"])
    pctiles2 = percentile_rank(radar_df, p2["player_id"], p2["position"])
    for dim in RADAR_DIMENSIONS:
        comparison_rows.append({
            "Matchup": f"{p1['full_name']} vs {p2['full_name']}",
            "Dimension": dim,
            p1["full_name"]: pctiles1[dim],
            p2["full_name"]: pctiles2[dim],
            "Advantage": p1["full_name"] if pctiles1[dim] > pctiles2[dim] else p2["full_name"],
        })

comp_df = pl.DataFrame(comparison_rows)
print("Dimension-by-dimension advantages (first matchup shown):")
comp_df.filter(
    pl.col("Matchup") == comp_df["Matchup"][0]
)

In [ ]:
takeaway(
    "Head-to-head radar comparisons highlight how even similarly-ranked players occupy "
    "different niches. Overlapping polygons show shared strengths; gaps reveal trade-offs "
    "and the dimensions where one player clearly outclasses the other."
)

---

## 8. Cleanup & Cross-Links

In [ ]:
# Cleanup handled by nbadb_utils.get_connection() via atexit
print("Analysis complete.")

### nbadb Notebook Suite

This is **Part 6 of 10**. Explore the full suite:

| Part | Notebook | Description |
|---|---|---|
| Part 1 | [MVP Prediction](./nba_mvp_predictor.ipynb) | MVP Prediction with Tracking & Synergy Data |
| Part 2 | [Data-Driven Player Archetypes](./nba_player_archetypes.ipynb) | Data-Driven Player Archetypes (UMAP + GMM) |
| Part 3 | [Game Outcome Prediction](./nba_game_prediction.ipynb) | Game Outcome Prediction (Stacking Ensemble) |
| Part 4 | [Draft Combine to Career](./nba_draft_combine_analysis.ipynb) | Draft Combine to Career Prediction |
| Part 5 | [Defense Decoded](./nba_defense_decoded.ipynb) | Defense Decoded (Tracking + Hustle + Synergy PCA) |
| **Part 6** | **Interactive Player Explorer** (this notebook) | **Interactive Player Explorer** |
| Part 7 | [Spatial Shot Analysis](./nba_shot_chart_analysis.ipynb) | Spatial Shot Analysis & 3-Point Revolution |
| Part 8 | [Player Similarity Engine](./nba_player_similarity.ipynb) | Player Similarity Engine (Cosine + Manhattan) |
| Part 9 | [Career Trajectory](./nba_aging_curves.ipynb) | Career Trajectory & Aging Curve Modeling |
| Part 10 | [Play-by-Play Insights](./nba_play_by_play_insights.ipynb) | Play-by-Play: Win Probability, Runs & Clutch |

---

[Dataset](https://www.kaggle.com/datasets/wyattowalsh/basketball) | [GitHub](https://github.com/wyattowalsh/nbadb) | [Documentation](https://nbadb.dev)